# Session 1: When Optimal Fails — Stress-Testing Classical Minimum-Variance Portfolios

Modern portfolio theory promises an "optimal" allocation — but optimal under what assumptions? In this session, we'll build a classical minimum-variance portfolio and then systematically break it. We'll use AI-driven "what-if" stress tests to expose how fragile textbook solutions can be when the world doesn't cooperate.

> **By the end of this session, you will be able to:**
> * Construct a classical minimum-variance portfolio from estimated return and covariance data
> * Identify the key weaknesses of mean-variance optimization: input sensitivity, weight concentration, and assumption fragility
> * Design and run AI-powered stress tests that shift correlations, simulate price drops, and increase trading costs
> * Produce a baseline performance scorecard (expected return, maximum drawdown, turnover, and trading cost) that we'll use to benchmark strategies in later sessions

Let's get started!

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

➡ [Example: Building a Classical Minimum-Variance Portfolio](eCornell-AI-Finance-L1-Example-BuildMinVariancePortfolio-May-2026.ipynb). In this example, we generate synthetic asset return data, estimate the expected return vector and covariance matrix, and solve the minimum-variance optimization problem. We'll visualize the resulting portfolio weights and explore their sensitivity to the input estimates.

➡ [Example: Stress-Testing the Minimum-Variance Portfolio](eCornell-AI-Finance-L1-Example-StressTestMinVariancePortfolio-May-2026.ipynb). In this example, we run systematic "what-if" experiments: shifting the correlation structure, simulating sudden price drops, and increasing trading costs. We compare the stress-tested portfolios to the baseline and produce a performance scorecard.

## The Classical Minimum-Variance Portfolio
In 1952, Harry Markowitz proposed a deceptively simple idea: investors care about the _expected return_ and the _variance_ (risk) of their portfolio, and they want the best trade-off between the two. The minimum-variance portfolio is the allocation that achieves the lowest possible risk for a given level of expected return.

> **The Setup.** Suppose we have $N$ assets with expected return vector $\boldsymbol{\mu}\in\mathbb{R}^{N}$ and covariance matrix $\boldsymbol{\Sigma}\in\mathbb{R}^{N\times N}$. We seek a weight vector $\mathbf{w}\in\mathbb{R}^{N}$ that solves:
>
> $$\min_{\mathbf{w}} \quad \mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}$$
>
> subject to:
>
> $$\mathbf{w}^{\top}\boldsymbol{\mu} \geq R, \qquad \sum_{i=1}^{N} w_{i} = 1, \qquad l_{i} \leq w_{i} \leq u_{i} \quad \forall\, i$$
>
> where $R$ is the target return and $l_i, u_i$ are the lower and upper bounds on each weight (e.g., no short selling means $l_i = 0$).

This is a convex quadratic program — it has a unique global minimum and can be solved efficiently. On the surface, it seems like the perfect recipe for rational investing. But there's a catch: _everything depends on the quality of $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$._

## When Optimal Fails: Three Weaknesses of Mean-Variance Optimization

The minimum-variance framework is elegant, but practitioners have learned — often painfully — that it has three systemic weaknesses:

### 1. Input Sensitivity
The optimizer treats $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ as _ground truth_. But these are estimated from historical data, and small estimation errors can produce wildly different optimal weights. 

> **The Estimation Problem.** Expected returns are notoriously difficult to estimate. A change of just 50 basis points in one asset's expected return can shift its optimal weight by 20% or more. The covariance matrix is more stable, but still subject to estimation error — especially off-diagonal terms (correlations) during periods of market stress.

This is sometimes called the "error maximization" property of mean-variance optimization: the optimizer aggressively exploits estimation errors, loading up on assets whose returns are _overestimated_ and avoiding those that are _underestimated_.

### 2. Weight Concentration
Without tight constraints, minimum-variance portfolios tend to concentrate in a small number of assets — often the ones with the lowest historical variance. This produces portfolios that _look_ optimal on paper but carry hidden concentration risk.

> **Why does this happen?** The quadratic objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}$ rewards low-variance assets disproportionately. If two assets have similar expected returns but one has slightly lower variance, the optimizer will heavily favor the lower-variance asset. In practice, this means a "diversified" portfolio can end up with 60-80% of its weight in just 2-3 names.

### 3. Assumption Fragility
The entire framework assumes that returns are drawn from a stationary multivariate distribution — that the future will look like the past. This assumption breaks during exactly the moments when portfolio construction matters most: regime changes, market crises, and structural shifts.

> **Stationarity is a luxury.** Correlations spike during crises (assets that were "diversifying" suddenly move together), volatilities jump, and expected returns shift. The portfolio that was "optimal" under calm conditions can become the _worst_ possible allocation when the regime changes.

## Stress Testing: Using AI to Break Your Own Portfolio

If the weakness of mean-variance optimization is its dependence on assumptions, the remedy is to _systematically violate those assumptions_ and see what happens. This is the idea behind stress testing.

We'll design three categories of "what-if" experiments:

> **Stress Test 1: Correlation Shifts.** What happens to our portfolio if pairwise correlations increase by 10%, 25%, or 50%? During crises, correlations tend to spike toward 1.0 — the diversification benefit we counted on evaporates. We perturb the correlation matrix $\mathbf{C}$ as:
>
> $$\tilde{\mathbf{C}} = (1-\alpha)\,\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}$$
>
> where $\alpha\in[0,1]$ controls the severity of the stress. At $\alpha = 0$ we have the original correlations; at $\alpha = 1$, all assets are perfectly correlated.

> **Stress Test 2: Price Drops.** What if one or more assets experience a sudden price decline? We simulate scenarios where individual assets or sectors drop by 10%, 20%, or 40%, and recompute the portfolio's performance. This tests the portfolio's _tail risk_ exposure.

> **Stress Test 3: Higher Trading Costs.** What if transaction costs are 2×, 5×, or 10× our baseline estimate? High-turnover portfolios that look optimal under low-cost assumptions can become net losers when costs rise — for instance during periods of low liquidity or market stress.

In each case, we re-solve the optimization under the stressed inputs and compare the resulting portfolio to the baseline. The AI's role here is to _automate the generation and evaluation_ of hundreds of stress scenarios, producing a comprehensive map of how the portfolio degrades.

## The Baseline Scorecard

Before we can evaluate any improvements (that's Sessions 2–4), we need a _baseline_ — a quantitative record of how the classical minimum-variance portfolio performs under both normal and stressed conditions. Our scorecard tracks four metrics:

| Metric | Definition | Why It Matters |
|--------|-----------|---------------|
| **Expected Return** | $\mathbf{w}^{\top}\boldsymbol{\mu}$ | Does the portfolio meet our return target? |
| **Maximum Drawdown** | Largest peak-to-trough decline in cumulative wealth | How bad does the worst loss get? |
| **Turnover** | $\sum_{i=1}^{N}\lvert w_{i,t} - w_{i,t-1}\rvert$ | How much trading is required to maintain the allocation? |
| **Trading Cost** | Turnover $\times$ cost-per-unit-traded | What is the real cost of rebalancing? |

> **Why these four?** Return tells us if the strategy is worth pursuing. Drawdown tells us if we can survive the journey. Turnover tells us how active the strategy is. Trading cost tells us if the activity is economically justified. Together, they form a _necessary and sufficient_ baseline for comparing strategies.

We'll compute this scorecard under the baseline assumptions and under each stress scenario. The result is a matrix of outcomes that reveals exactly where the classical approach breaks down — and that sets the stage for Session 2, where we'll design an AI-powered rebalancing engine to address these weaknesses.

## Summary

> **Key Takeaways from Session 1:**
> * The classical minimum-variance portfolio solves a clean quadratic program, but its solution quality is _entirely dependent_ on the accuracy of estimated returns $\boldsymbol{\mu}$ and covariances $\boldsymbol{\Sigma}$
> * Three systemic weaknesses — input sensitivity, weight concentration, and assumption fragility — mean that "optimal" portfolios can fail precisely when they are needed most
> * Systematic stress testing (correlation shifts, price drops, cost increases) reveals the _boundary conditions_ under which the classical approach degrades
> * The baseline scorecard (return, drawdown, turnover, trading cost) gives us a quantitative benchmark that we'll use throughout the remaining sessions to evaluate AI-driven improvements

**Next Session:** In [Session 2](../session-2/L2/eCornell-AI-Finance-L2-Lecture-AIRebalancingEngine-May-2026.ipynb), we'll move from static weights to adaptive allocation — designing an AI rebalancing engine that incorporates sentiment, fundamentals, and technical signals to respond dynamically to changing market conditions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models. Real-world portfolio construction involves additional considerations including regulatory requirements, tax implications, and operational constraints that are beyond the scope of this session.